In [1]:
import kaggle 
import pandas as pd 
import numpy as np 
from scipy.stats import stats

In [2]:
df = pd.read_excel(r"C:\Users\RITHIK KUMAR\E Commerce Dataset.xlsx", sheet_name="E Comm")
df.shape

(5630, 20)

In [3]:
df.head()

,CustomerID,Churn,Tenure,PreferredLoginDevice,CityTier,WarehouseToHome,PreferredPaymentMode,Gender,HourSpendOnApp,NumberOfDeviceRegistered,PreferedOrderCat,SatisfactionScore,MaritalStatus,NumberOfAddress,Complain,OrderAmountHikeFromlastYear,CouponUsed,OrderCount,DaySinceLastOrder,CashbackAmount
0,50001,1,4.0,Mobile Phone,3,6.0,Debit Card,Female,3.0,3,Laptop & Accessory,2,Single,9,1,11.0,1.0,1.0,5.0,159.93
1,50002,1,NaN,Phone,1,8.0,UPI,Male,3.0,4,Mobile,3,Single,7,1,15.0,0.0,1.0,0.0,120.90
2,50003,1,NaN,Phone,1,30.0,Debit Card,Male,2.0,4,Mobile,3,Single,6,1,14.0,0.0,1.0,3.0,120.28
3,50004,1,0.0,Phone,3,15.0,Debit Card,Male,2.0,4,Laptop & Accessory,5,Single,8,0,23.0,0.0,1.0,3.0,134.07
4,50005,1,0.0,Phone,1,12.0,CC,Male,NaN,3,Mobile,5,Single,3,0,11.0,1.0,1.0,3.0,129.60


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5630 entries, 0 to 5629
Data columns (total 20 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   CustomerID                   5630 non-null   int64  
 1   Churn                        5630 non-null   int64  
 2   Tenure                       5366 non-null   float64
 3   PreferredLoginDevice         5630 non-null   object 
 4   CityTier                     5630 non-null   int64  
 5   WarehouseToHome              5379 non-null   float64
 6   PreferredPaymentMode         5630 non-null   object 
 7   Gender                       5630 non-null   object 
 8   HourSpendOnApp               5375 non-null   float64
 9   NumberOfDeviceRegistered     5630 non-null   int64  
 10  PreferedOrderCat             5630 non-null   object 
 11  SatisfactionScore            5630 non-null   int64  
 12  MaritalStatus                5630 non-null   object 
 13  NumberOfAddress   

In [5]:
df.isna().sum()

CustomerID                       0
Churn                            0
Tenure                         264
PreferredLoginDevice             0
CityTier                         0
WarehouseToHome                251
PreferredPaymentMode             0
Gender                           0
HourSpendOnApp                 255
NumberOfDeviceRegistered         0
PreferedOrderCat                 0
SatisfactionScore                0
MaritalStatus                    0
NumberOfAddress                  0
Complain                         0
OrderAmountHikeFromlastYear    265
CouponUsed                     256
OrderCount                     258
DaySinceLastOrder              307
CashbackAmount                   0
dtype: int64

In [6]:
df[df["Tenure"].isna() & df["Churn"] == 1]["Churn"].count()

81

In [7]:
df[df["Tenure"].isna()]["Churn"].value_counts(normalize=True) * 100

Churn
0    69.318182
1    30.681818
Name: proportion, dtype: float64

In [8]:
df["Churn"].value_counts(normalize=True) * 100

Churn
0    83.161634
1    16.838366
Name: proportion, dtype: float64

## Finding 1: Missing Tenure is correlated with Churn

- Overall churn rate: 16.8%
- Churn rate among missing-Tenure customers: 30.7% (81/264)
- Missing Tenure customers churn at ~2x the baseline rate
- Missingness is NOT random — likely early dropoffs with incomplete records
- Action: Create `Tenure_missing` flag before imputation to preserve this signal

In [9]:
df[df["Churn"] == 1]["Complain"].value_counts(normalize=True) * 100

Complain
1    53.586498
0    46.413502
Name: proportion, dtype: float64

In [10]:
df.groupby("Churn")["Complain"].value_counts(normalize=True) * 100

Churn  Complain
0      0           76.591200
       1           23.408800
1      1           53.586498
       0           46.413502
Name: proportion, dtype: float64

## Finding 2: Complaints are strongly associated with churn

- 53.6% of churned customers had raised a complaint
- Only 23.4% of retained customers raised a complaint
- Churned customers complain at 2.3x the rate of retained customers
- Action: Complaint handling is a high-leverage retention intervention point

In [11]:
df.groupby("Gender")["Churn"].value_counts(normalize=True)

Gender  Churn
Female  0        0.845058
        1        0.154942
Male    0        0.822695
        1        0.177305
Name: proportion, dtype: float64

## Finding 3: Gender has minimal impact on churn

- Male churn rate: 17.7%, Female churn rate: 15.5%
- Difference is only 2.2pp — not a meaningful segmentation variable
- Gender alone is not a useful retention targeting dimension

In [12]:
df["Tenure_bin"] = pd.cut(df["Tenure"], bins=[0, 6, 12, 24, 62], labels=["0-6m", "6-12m", "12-24m", "24m+"])
df.groupby("Tenure_bin")["Churn"].mean() * 100

C:\Users\RITHIK KUMAR\AppData\Local\Temp\ipykernel_19092\4181655823.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby("Tenure_bin")["Churn"].mean() * 100


Tenure_bin
0-6m      25.883069
6-12m      5.681818
12-24m     6.475801
24m+       0.000000
Name: Churn, dtype: float64

## Finding 4: Churn is heavily concentrated in the first 6 months

- 0-6 month customers churn at 25.9% vs 5-6% for longer tenure
- Customers surviving past 24 months show 0% churn
- The first 6 months is the critical retention window
- Business action: Onboarding experience and early engagement programs 
  are the highest ROI retention investment

In [13]:
pd.set_option("display.max_columns", None)
df.head()

,CustomerID,Churn,Tenure,PreferredLoginDevice,CityTier,WarehouseToHome,PreferredPaymentMode,Gender,HourSpendOnApp,NumberOfDeviceRegistered,PreferedOrderCat,SatisfactionScore,MaritalStatus,NumberOfAddress,Complain,OrderAmountHikeFromlastYear,CouponUsed,OrderCount,DaySinceLastOrder,CashbackAmount,Tenure_bin
0,50001,1,4.0,Mobile Phone,3,6.0,Debit Card,Female,3.0,3,Laptop & Accessory,2,Single,9,1,11.0,1.0,1.0,5.0,159.93,0-6m
1,50002,1,NaN,Phone,1,8.0,UPI,Male,3.0,4,Mobile,3,Single,7,1,15.0,0.0,1.0,0.0,120.90,NaN
2,50003,1,NaN,Phone,1,30.0,Debit Card,Male,2.0,4,Mobile,3,Single,6,1,14.0,0.0,1.0,3.0,120.28,NaN
3,50004,1,0.0,Phone,3,15.0,Debit Card,Male,2.0,4,Laptop & Accessory,5,Single,8,0,23.0,0.0,1.0,3.0,134.07,NaN
4,50005,1,0.0,Phone,1,12.0,CC,Male,NaN,3,Mobile,5,Single,3,0,11.0,1.0,1.0,3.0,129.60,NaN


In [14]:
df.groupby("SatisfactionScore")["Churn"].mean() * 100

SatisfactionScore
1    11.512027
2    12.627986
3    17.196702
4    17.132216
5    23.826715
Name: Churn, dtype: float64

## Finding 5: Satisfaction score shows counterintuitive churn pattern

- Churn rate increases with satisfaction score (11.5% at score 1 vs 23.8% at score 5)
- Counterintuitive — suggests score may be measuring something other than loyalty
- Possible explanations: score captured post-churn, or satisfied customers 
  are more willing to leave for better alternatives
- Needs further investigation before using as a model feature

In [15]:
to_Check = ["DaySinceLastOrder", "CouponUsed","NumberOfDeviceRegistered","CityTier","WarehouseToHome"]

for item in to_Check:
    print(f"Churn analysis of {item}:")
    print(df.groupby(item)["Churn"].mean() * 100)
    print()

Churn analysis of DaySinceLastOrder:
DaySinceLastOrder
0.0      34.274194
1.0      29.641694
2.0      15.404040
3.0      14.444444
4.0      13.457077
5.0      12.280702
6.0      14.159292
7.0      14.541387
8.0      12.267658
9.0       9.364548
10.0      6.369427
11.0      8.791209
12.0      2.898551
13.0      0.000000
14.0     11.428571
15.0     21.052632
16.0      0.000000
17.0      0.000000
18.0      0.000000
30.0      0.000000
31.0      0.000000
46.0    100.000000
Name: Churn, dtype: float64

Churn analysis of CouponUsed:
CouponUsed
0.0      18.058252
1.0      17.957245
2.0      17.303196
3.0      14.067278
4.0      15.228426
5.0      17.829457
6.0      16.666667
7.0      20.224719
8.0      21.428571
9.0      15.384615
10.0     21.428571
11.0     16.666667
12.0     11.111111
13.0      0.000000
14.0      0.000000
15.0    100.000000
16.0     50.000000
Name: Churn, dtype: float64

Churn analysis of NumberOfDeviceRegistered:
NumberOfDeviceRegistered
1     9.361702
2     9.420290
3    1

## Finding 6: Recency (DaySinceLastOrder) shows U-shaped churn pattern

- Customers with 0-1 days since last order churn at 30-34% — highest risk group
- Churn drops to 2-6% at 10-13 days — lowest risk window
- Churn rises again beyond 15+ days — disengagement signal
- Pattern is U-shaped, not linear — recency alone is insufficient, direction matters
- Business action: Investigate 0-1 day churners — likely one-time buyers or 
  post-purchase regret; re-engage 15+ day inactive customers before they leave

## Finding 7: Number of Devices Registered is a monotonic churn predictor

- 1-2 devices: ~9.4% churn (lowest risk)
- 6 devices: 34.6% churn (highest risk)
- Clear monotonic increase — every additional device raises churn probability
- Likely indicates comparison shoppers active across multiple platforms
- Business action: High device count customers need loyalty incentives to anchor them

## Finding 8: Warehouse-to-Home distance has a churn threshold effect

- Below 15km: churn stable at 12-13%
- Beyond 28km: churn jumps to 25-33%
- Logistics friction becomes a churn driver beyond a distance threshold
- Business action: Priority fast delivery or delivery subsidies for customers 
  beyond 25km radius

In [16]:
def get_list_od_missing_value_columns(df):
    length = len(df)
    missing_columns = {}
    columns = df.columns
    for col in columns:
        missing  = df[col].isna().sum()
        if missing > 0 :
            prct = (missing/length) * 100
            missing_columns[col] = round(prct,2)
    return missing_columns

In [17]:
result = get_list_od_missing_value_columns(df)

In [18]:
df["Tenure_missing"] = df["Tenure"].isna().astype(int)

In [19]:
import plotly.graph_objects as go 


def plot_distribution(col):
    fig = go.Figure()

    fig.add_trace(
        go.Histogram(
            x = df[col],
            nbinsx= 15,
            marker=dict(color = 'skyblue', line = dict(color = 'black', width = 1))
        )
    )
    fig.update_layout(
        xaxis_title = f"{col}_Distribution",
        yaxis_title = "Frequency",
        title = f"Histogram Distriubtion of {col}"
    )
    fig.show()
    

In [20]:
for key,value in result.items():
    plot_distribution(key)

In [21]:
df["WarehouseToHome"].describe()

count    5379.000000
mean       15.639896
std         8.531475
min         5.000000
25%         9.000000
50%        14.000000
75%        20.000000
max       127.000000
Name: WarehouseToHome, dtype: float64

In [24]:
df = df[df["WarehouseToHome"] <=100]

In [25]:
from scipy.stats import mannwhitneyu, chi2_contingency

churned  = df[df["Churn"] == 1]["Tenure"].dropna()
retained = df[df["Churn"] == 0]["Tenure"].dropna()

stat, p = mannwhitneyu(churned,retained, alternative="two-sided")
print(f"p-value : {p:.4f}")

p-value : 0.0000


In [26]:
threshod = 0.05 

if  p < threshod:
    print("There is statistical difference present")
else :
    print("There is no enough evidence for H0")

There is statistical difference present


In [27]:
n1, n2 = len(churned), len(retained)
r = 1 - (2 * stat) / (n1 * n2)
print(f"Effect size (rank-biserial r): {r:.4f}")

Effect size (rank-biserial r): 0.6378


In [28]:
def find_statistical_difference_mannwhite(col_name):
    churned  = df[df["Churn"] == 1][col_name].dropna()
    retained = df[df["Churn"] == 0][col_name].dropna()
    stat, p = mannwhitneyu(churned,retained, alternative="two-sided")
    print(f"p-value of {col_name} : {p:.4f}")
    n1, n2 = len(churned), len(retained)
    r = 1 - (2 * stat) / (n1 * n2)
    print(f"Effect size (rank-biserial r): {r:.4f}")
    threshod = 0.05 

    if  p < threshod:
        return("There is statistical difference present")
    else :
        return("There is no enough evidence for H0")

In [32]:
columns = ["Tenure","SatisfactionScore","NumberOfDeviceRegistered","WarehouseToHome"]
for col in columns:
    print("="*40)
    print(find_statistical_difference_mannwhite(col))
    print("="*40)
    print()

p-value of Tenure : 0.0000
Effect size (rank-biserial r): 0.6378
There is statistical difference present

p-value of SatisfactionScore : 0.0000
Effect size (rank-biserial r): -0.1568
There is statistical difference present

p-value of NumberOfDeviceRegistered : 0.0000
Effect size (rank-biserial r): -0.1585
There is statistical difference present

p-value of WarehouseToHome : 0.0000
Effect size (rank-biserial r): -0.1279
There is statistical difference present



In [30]:
df["Tenure_bin"]

0         0-6m
1          NaN
2          NaN
3          NaN
4          NaN
         ...  
5625     6-12m
5626    12-24m
5627      0-6m
5628    12-24m
5629     6-12m
Name: Tenure_bin, Length: 5377, dtype: category
Categories (4, object): ['0-6m' < '6-12m' < '12-24m' < '24m+']

In [33]:
df.columns

Index(['CustomerID', 'Churn', 'Tenure', 'PreferredLoginDevice', 'CityTier',
       'WarehouseToHome', 'PreferredPaymentMode', 'Gender', 'HourSpendOnApp',
       'NumberOfDeviceRegistered', 'PreferedOrderCat', 'SatisfactionScore',
       'MaritalStatus', 'NumberOfAddress', 'Complain',
       'OrderAmountHikeFromlastYear', 'CouponUsed', 'OrderCount',
       'DaySinceLastOrder', 'CashbackAmount', 'Tenure_bin', 'Tenure_missing'],
      dtype='object')

In [42]:
def chi_square(col):
    ct = pd.crosstab(df[col],df["Churn"])
    stat,p,dof,expected = chi2_contingency(ct)
    n = ct.values.sum()
    k =  min(ct.shape)

    cramers_v = np.sqrt(stat / (n * (k-1)))
    print(f"Cramers's V of {col} : ", cramers_v)
    threshod = 0.05 

    if  p < threshod:
        return("There is statistical difference present")
    else :
        return("There is no enough evidence for H0")

In [43]:
columns = ["Complain","Tenure_bin"]

for col in columns:
    print("="*40)
    print(chi_square(col))
    print("="*40)
    print()

Cramers's V of Complain :  0.23804226039497128
There is statistical difference present

Cramers's V of Tenure_bin :  0.31294174977756706
There is statistical difference present



## Statistical Validation Summary

All six features were tested for statistically significant association with churn.
Significance threshold: p < 0.05

### Test Results

| Variable | Test Used | p-value | Effect Size | Strength |
|---|---|---|---|---|
| Tenure | Mann-Whitney U | 0.0000 | r = 0.64 | Large |
| Tenure_bin | Chi-square | 0.0000 | V = 0.31 | Moderate |
| Complain | Chi-square | 0.0000 | V = 0.24 | Moderate |
| SatisfactionScore | Mann-Whitney U | 0.0000 | r = -0.16 | Small |
| NumberOfDeviceRegistered | Mann-Whitney U | 0.0000 | r = -0.16 | Small |
| WarehouseToHome | Mann-Whitney U | 0.0000 | r = -0.13 | Small |

### Interpretation

All six variables show statistically significant association with churn (p < 0.05).
However, statistical significance alone does not indicate business importance —
effect size determines how strongly each variable actually drives churn.

- Tenure is the dominant driver with a large effect size (r = 0.64). Churned customers
  have significantly lower tenure than retained customers. The first 6 months is the
  critical risk window.

- Complain and Tenure_bin show moderate effect sizes (V = 0.24 and V = 0.31).
  Both are reliable, actionable signals — complaint history and early tenure stage
  are meaningful retention intervention points.

- SatisfactionScore, NumberOfDeviceRegistered, and WarehouseToHome show small
  effect sizes (r < 0.17). Statistically real but individually weak — useful as
  secondary signals in a composite risk score, not standalone drivers.

### Decision

Primary risk bracket drivers: Tenure_bin, Complain
Secondary signals: SatisfactionScore, NumberOfDeviceRegistered, WarehouseToHome
Excluded from brackets: None — all retained as supporting features